# WithU Talk — Phase 0–1: Unified Label Schema & Event-Level Splits

This notebook prepares the training/evaluation data for the WithU Talk cyberbullying
detector, following the revised plan:

- **Public corpora (BEEP! + K-MHaS + AI-Hub)** → *general-toxicity* training pool (Stage-1).
- **WithU Excel** → *domain-adaptation seed* + **evaluation gold**, split **at the event level**.

It does three things and nothing else (keep it boring and correct — everything downstream sits on it):

1. Define one **unified label schema** that all sources map into.
2. **Normalize** BEEP, K-MHaS, AI-Hub, and the WithU Excel into that schema (with cleaning + quarantine).
3. Produce a **leak-free event-level split** of the WithU data + a stratified split of the public pool, and save to disk.

> The loaders below were tested against the real BEEP `train/dev.tsv`, real K-MHaS
> `kmhas_train.txt` (label `8` = *not hate*), and your actual Excel (52,767 rows, 21 cols).


## 0. Files you need to download

| Source | What / size | How to get it | Where to put it |
|---|---|---|---|
| **Korean HateSpeech (BEEP!)** | ~9.4K labeled news comments (`hate`/`offensive`/`none`) | `git clone https://github.com/kocohub/korean-hate-speech` | `data/korean-hate-speech/labeled/train.tsv`, `dev.tsv` |
| **K-MHaS** | 109K news comments, multi-label ints (`8`=not-hate) | `git clone https://github.com/adlnlp/K-MHaS` | `data/K-MHaS/data/kmhas_{train,valid,test}.txt` |
| **AI-Hub sets** | 텍스트 윤리검증 / 속성기반 감정분석 / SNS 멀티턴 등 | **Manual** — requires an AI-Hub account + dataset application at https://aihub.or.kr (cannot be scripted) | `data/aihub/<set_name>/...` |
| **WithU Excel** | your labeled corpus | (you already have it) | set `WITHU_XLSX` below |

The two GitHub sets can be auto-fetched by the next cell (`AUTO_DOWNLOAD = True`). AI-Hub must be
downloaded by hand because of its login/approval wall; a pluggable stub loader is provided so you can
drop those files in later without changing anything else.

*HF alternative:* if you prefer, BEEP is on the Hub and K-MHaS is at
`jeanlee/kmhas_korean_hate_speech` via `datasets.load_dataset(...)`. This notebook uses the local
files so it runs fully offline once downloaded.


pip install pandas numpy scikit-learn openpyxl pyarrow

In [3]:
import os, re, subprocess, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
pd.set_option("display.max_colwidth", 60)

In [34]:
# ---------------------------------------------------------------------------
# CONFIG — edit these paths, then run top to bottom.
# ---------------------------------------------------------------------------
DATA_DIR   = Path("./data")            # where public datasets live / will be cloned
OUT_DIR    = Path("./prepared")        # where normalized splits are written
WITHU_XLSX = Path("/home/yonsei/Downloads/excel_wangtta.xlsx")   # <-- your Excel
WITHU_SHEET = "sheet1"

BEEP_DIR   = DATA_DIR / "korean-hate-speech" / "labeled"
KMHAS_DIR  = DATA_DIR / "K-MHaS" / "data"
AIHUB_DIR  = DATA_DIR / "aihub"        # manual downloads go here

AUTO_DOWNLOAD = True                   # git clone BEEP + K-MHaS if missing

# Split knobs (see the warning in the split section — event count is TINY)
N_TEST_CB_EVENTS = 1
N_VAL_CB_EVENTS  = 1
SEED = 42

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
def _clone(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f"  [skip] {dest} already present"); return
    print(f"  [clone] {url} -> {dest}")
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)

if AUTO_DOWNLOAD:
    try:
        _clone("https://github.com/kocohub/korean-hate-speech",
               DATA_DIR / "korean-hate-speech")
        _clone("https://github.com/adlnlp/K-MHaS",
               DATA_DIR / "K-MHaS")
    except Exception as e:
        warnings.warn(f"Auto-download failed ({e}). Download the files manually per the table above.")

print("BEEP train exists:", (BEEP_DIR / "train.tsv").exists())
print("K-MHaS train exists:", (KMHAS_DIR / "kmhas_train.txt").exists())
print("WithU xlsx exists:", WITHU_XLSX.exists())

  [skip] data/korean-hate-speech already present
  [skip] data/K-MHaS already present
BEEP train exists: True
K-MHaS train exists: True
WithU xlsx exists: True


## 1. The unified label schema

Every row from every source becomes one record with these fields. Public rows leave the
domain-only fields empty; the WithU rows fill everything.

- `text` — the utterance / comment
- `source` — `beep` | `kmhas` | `aihub` | `withu`
- `cb_binary` — 1 = toxic/cyberbullying present, 0 = clean, `NaN` = undecided (quarantined)
- `cb_type` — `비해당` | `언어적 폭력` | `시각적 폭력` | `배제` | composites (`언어적 폭력·배제`)
- `is_verbal` / `is_visual` / `is_exclusion` — booleans (support multi-type events)
- `bystander` — `방어` | `동조` | `방관` | `None`  *(WithU only)*
- `event_id` — conversation/event id *(WithU only; drives the leak-free split)*
- `speaker` — *(WithU only)*
- `quarantine` / `quarantine_reason` — rows with `검토`/`판정불가` labels held out of clean sets
- `split` — assigned later

**Label-mapping decisions (be explicit so reviewers can audit):**
- BEEP `hate` + `offensive` → toxic (`언어적 폭력`); `none` → clean.
- K-MHaS label set `== {8}` → clean; any of `0–7` present → toxic (`언어적 폭력`).
- These map hate-speech to the *verbal-violence* type only. The public sets carry **no**
  visual/exclusion/bystander signal — those come from WithU + synthetic data in later phases.


In [36]:
UNIFIED_COLS = ["text","source","cb_binary","cb_type",
                "is_verbal","is_visual","is_exclusion",
                "bystander","event_id","speaker",
                "quarantine","quarantine_reason","split"]

def _empty_domain_fields(df, cb):
    df["is_verbal"]    = cb == 1
    df["is_visual"]    = False
    df["is_exclusion"] = False
    df["bystander"]    = None
    df["event_id"]     = None
    df["speaker"]      = None
    df["quarantine"]   = False
    df["quarantine_reason"] = ""
    return df

## 2a. BEEP! (Korean HateSpeech) loader

In [37]:
def load_beep(path, source="beep"):
    """Columns: comments, contain_gender_bias, bias, hate ; hate in {hate,offensive,none}."""
    df = pd.read_csv(path, sep="\t")
    hate = df["hate"].astype(str).str.strip().str.lower()
    cb = hate.isin(["hate", "offensive"]).astype(int)
    out = pd.DataFrame({
        "text": df["comments"].astype(str),
        "source": source,
        "cb_binary": cb,
        "cb_type": np.where(cb == 1, "언어적 폭력", "비해당"),
    })
    return _empty_domain_fields(out, cb)

## 2b. K-MHaS loader

In [38]:
def load_kmhas(path, source="kmhas", not_hate_label="8"):
    """Columns: document, label ; label = comma-separated ints. {8}=not hate, 0-7=hate types."""
    df = pd.read_csv(path, sep="\t")
    def is_toxic(lab):
        toks = [t.strip() for t in str(lab).split(",")]
        return 0 if toks == [not_hate_label] else 1
    cb = df["label"].map(is_toxic).astype(int)
    out = pd.DataFrame({
        "text": df["document"].astype(str),
        "source": source,
        "cb_binary": cb,
        "cb_type": np.where(cb == 1, "언어적 폭력", "비해당"),
    })
    return _empty_domain_fields(out, cb)

## 2c. AI-Hub loader (pluggable stub)

AI-Hub sets vary in structure and sit behind a login/approval wall, so they can't be auto-loaded.
Fill in `load_aihub` per set once you've downloaded them (map each set's toxic/clean field to
`cb_binary`, text to `text`). Until then this returns an empty frame and the pipeline runs fine
on BEEP + K-MHaS alone.


In [39]:
def load_aihub(aihub_dir=AIHUB_DIR, source="aihub"):
    """Return unified-schema rows from AI-Hub sets. Implement per downloaded set.

    Example skeleton for a set with a 'sentence' + 'is_toxic' style file:
        rows = []
        for f in Path(aihub_dir).glob('text_ethics/*.json'):
            data = json.loads(Path(f).read_text(encoding='utf-8'))
            for item in data:
                rows.append({'text': item['sentence'],
                             'cb_binary': int(item['label'] != '정상')})
        df = pd.DataFrame(rows); df['source']=source
        df['cb_type']=np.where(df.cb_binary==1,'언어적 폭력','비해당')
        return _empty_domain_fields(df, df['cb_binary'])
    """
    d = Path(aihub_dir)
    if not d.exists() or not any(d.iterdir()):
        warnings.warn("AI-Hub dir empty — skipping (implement load_aihub after manual download).")
        return pd.DataFrame(columns=UNIFIED_COLS)
    raise NotImplementedError("Implement AI-Hub parsing for your downloaded sets.")

## 2d. WithU Excel normalizer

Applies the cleaning rules we settled on:
- `사건_CB라벨` → `cb_binary`: `1`→1, `0`/`0(1차)`→0, `검토`→**quarantine** (undecided).
- `사이버불링_유형` → typed booleans + a `cb_type` label (handles composites like `언어적 폭력·배제`).
- `주변인행동` → `bystander` (`동조`/`방어`/`방관`); `판정불가`/`사건내-판정불가`/long free-text → **quarantine**.
- `사건ID` → `event_id` (drives the leak-free split); `발화자`, `행유형` retained.

`행유형 == system` rows (auto-notices) are kept but flagged out of the text-training view.


In [40]:
def _norm_binary(v):
    s = str(v).strip()
    if s == "1":            return 1,  False, ""
    if s in ("0","0(1차)"): return 0,  False, ""
    if s == "검토":         return np.nan, True, "라벨=검토(미확정)"
    return np.nan, True, f"라벨판독불가:{s}"

def _norm_cbtype(v):
    s = str(v).strip()
    verbal    = "언어적 폭력" in s
    visual    = "시각" in s
    exclusion = "배제" in s
    if s.startswith("비해당") or s in ("None","nan",""):
        label = "비해당"
    else:
        parts = []
        if verbal: parts.append("언어적 폭력")
        if visual: parts.append("시각적 폭력")
        if exclusion: parts.append("배제")
        label = "·".join(parts) if parts else "비해당"
    return label, verbal, visual, exclusion, ("후보" in s)

def _norm_bystander(v):
    s = str(v).strip()
    if s in ("동조","방어","방관"):          return s, False, ""
    if s == "비해당":                        return None, False, ""
    if s in ("판정불가","사건내-판정불가") or len(s) > 8:
        return None, True, f"주변인행동 미확정:{s[:20]}"
    return None, False, ""

def load_withu(path=WITHU_XLSX, sheet=WITHU_SHEET, source="withu"):
    raw = pd.read_excel(path, sheet_name=sheet, engine="openpyxl")
    col = {c: raw.iloc[:, i] for i, c in enumerate(raw.columns)}  # by-position, robust to renames
    cb  = col["사건_CB라벨"].map(_norm_binary)
    cbt = col["사이버불링_유형"].map(_norm_cbtype)
    by  = col["주변인행동"].map(_norm_bystander)

    df = pd.DataFrame({
        "text":         col["내용"].astype(str),
        "source":       source,
        "cb_binary":    [x[0] for x in cb],
        "cb_type":      [x[0] for x in cbt],
        "is_verbal":    [x[1] for x in cbt],
        "is_visual":    [x[2] for x in cbt],
        "is_exclusion": [x[3] for x in cbt],
        "is_candidate": [x[4] for x in cbt],
        "bystander":    [x[0] for x in by],
        "event_id":     col["사건ID"].where(col["사건ID"].notna(), None),
        "speaker":      col["발화자"],
        "row_type":     col["행유형"].astype(str),
    })
    q_cb = pd.Series([x[1] for x in cb]); qr_cb = pd.Series([x[2] for x in cb])
    q_by = pd.Series([x[1] for x in by]); qr_by = pd.Series([x[2] for x in by])
    df["quarantine"] = (q_cb | q_by).values
    df["quarantine_reason"] = [" | ".join(r for r in (a, b) if r) for a, b in zip(qr_cb, qr_by)]
    return df

## 3. Event-level split (leak-free) — **read the warning**

With augmentation and multi-turn events, splitting by *row* leaks copies of the same event into
train **and** test, inflating F1 into meaninglessness. So we hold out **whole events**.

Because all positives live in only **5 cyberbullying events (WCB001–005)**, we stratify the hold-out
on *CB-positive-bearing events* (so val/test actually contain positives), send other events + all
no-event background negatives into the pool, and distribute a slice of background into val/test for a
realistic negative volume.

> ⚠️ **This is a hard limit, not a code smell.** One event in val and one in test means your
> domain metrics are **indicative only** — report them with wide error bars, and treat the public
> test sets as the statistically-powered check for the text detector. This is exactly why Phases 3–4
> need more collected/synthetic events.


In [41]:
def event_level_split(df, n_test_cb_events=N_TEST_CB_EVENTS,
                      n_val_cb_events=N_VAL_CB_EVENTS,
                      bg_test_frac=0.15, bg_val_frac=0.15, seed=SEED):
    df = df.copy(); df["split"] = "domain_train"
    rng = np.random.default_rng(seed)
    ev_rows = df[df["event_id"].notna()]

    cb_events = sorted(ev_rows.groupby("event_id")["cb_binary"].max()
                              .pipe(lambda s: s[s == 1]).index)
    cb_shuf = list(cb_events); rng.shuffle(cb_shuf)
    if n_test_cb_events + n_val_cb_events >= len(cb_shuf):
        warnings.warn(f"Only {len(cb_shuf)} CB-positive events; holding out "
                      f"{n_test_cb_events+n_val_cb_events}. Test/val ≈ 1 event each — "
                      "metrics INDICATIVE ONLY.")
    test_ev = set(cb_shuf[:n_test_cb_events])
    val_ev  = set(cb_shuf[n_test_cb_events:n_test_cb_events+n_val_cb_events])
    df.loc[df["event_id"].isin(test_ev), "split"] = "domain_test"
    df.loc[df["event_id"].isin(val_ev),  "split"] = "domain_val"

    bg = df[df["event_id"].isna()].index.to_numpy().copy(); rng.shuffle(bg)
    n = len(bg); nte = int(n*bg_test_frac); nva = int(n*bg_val_frac)
    df.loc[bg[:nte], "split"]          = "domain_test"
    df.loc[bg[nte:nte+nva], "split"]   = "domain_val"

    meta = {"cb_events": cb_events,
            "test_events": sorted(test_ev), "val_events": sorted(val_ev),
            "other_events": sorted(set(ev_rows["event_id"].unique()) - set(cb_events))}
    return df, meta

def stratified_public_split(df, val_frac=0.1, test_frac=0.1, seed=SEED):
    """Independent rows → ordinary stratified split on cb_binary."""
    df = df.copy()
    tr, tmp = train_test_split(df, test_size=val_frac+test_frac,
                               stratify=df["cb_binary"], random_state=seed)
    rel = test_frac/(val_frac+test_frac)
    va, te = train_test_split(tmp, test_size=rel,
                              stratify=tmp["cb_binary"], random_state=seed)
    df.loc[tr.index, "split"] = "stage1_train"
    df.loc[va.index, "split"] = "stage1_val"
    df.loc[te.index, "split"] = "stage1_test"
    return df

## 4. Run it: normalize → split → save

In [42]:
# ---- public pool (Stage-1) ----
public_parts = []
if (BEEP_DIR/"train.tsv").exists():
    public_parts += [load_beep(BEEP_DIR/"train.tsv"), load_beep(BEEP_DIR/"dev.tsv")]
if (KMHAS_DIR/"kmhas_train.txt").exists():
    public_parts += [load_kmhas(KMHAS_DIR/"kmhas_train.txt"),
                     load_kmhas(KMHAS_DIR/"kmhas_valid.txt"),
                     load_kmhas(KMHAS_DIR/"kmhas_test.txt")]
public_parts.append(load_aihub())            # empty until you implement it
public = pd.concat([p for p in public_parts if len(p)], ignore_index=True)
public = public.drop_duplicates(subset=["text","source"]).reset_index(drop=True)
public = stratified_public_split(public)

print("PUBLIC pool:", len(public), "rows")
print(public.groupby(["source","split"]).agg(n=("text","size"),
      toxic=("cb_binary", lambda s:int(s.sum()))).to_string())

PUBLIC pool: 117206 rows
                         n  toxic
source split                     
beep   stage1_test     848    494
       stage1_train   6644   3735
       stage1_val      875    492
kmhas  stage1_test   10873   4909
       stage1_train  87120  39489
       stage1_val    10846   4912


/tmp/ipykernel_178468/1502333983.py:17: UserWarning: AI-Hub dir empty — skipping (implement load_aihub after manual download).
  warnings.warn("AI-Hub dir empty — skipping (implement load_aihub after manual download).")


In [43]:
# ---- WithU domain ----
withu_full = load_withu()
withu_clean = withu_full[(~withu_full.quarantine) & (withu_full.row_type == "message")].copy()
withu_split, meta = event_level_split(withu_clean)

print("Quarantined rows:", int(withu_full.quarantine.sum()))
print("Clean message rows:", len(withu_clean))
print("Split events:", json.dumps(meta, ensure_ascii=False))
print()
summary = withu_split.groupby("split").agg(
    n=("text","size"),
    toxic=("cb_binary", lambda s:int((s==1).sum())),
    defend=("bystander", lambda s:int((s=="방어").sum())),
    reinforce=("bystander", lambda s:int((s=="동조").sum())),
    bystand=("bystander", lambda s:int((s=="방관").sum())),
)
print(summary.to_string())

Quarantined rows: 609
Clean message rows: 51434
Split events: {"cb_events": ["WCB001", "WCB002", "WCB003", "WCB004", "WCB005"], "test_events": ["WCB005"], "val_events": ["WCB003"], "other_events": ["WEX001", "WEX002", "WEX003", "WRV001", "WRV003"]}

                  n  toxic  defend  reinforce  bystand
split                                                 
domain_test    7631     32       0          5        0
domain_train  36166    467      12         36        3
domain_val     7637     38       2          2        7


In [44]:
# ---- save artifacts ----
def _save(df, name):
    cols = [c for c in df.columns if c in set(UNIFIED_COLS)|{"is_candidate","row_type"}]
    df = df[cols]
    try:
        df.to_parquet(OUT_DIR/f"{name}.parquet", index=False)
        path = OUT_DIR/f"{name}.parquet"
    except Exception:
        path = OUT_DIR/f"{name}.csv"; df.to_csv(path, index=False)
    print(f"  saved {path}  ({len(df)} rows)")
    return path

print("Writing to", OUT_DIR.resolve())
_save(public, "stage1_public")                                   # general-toxicity pool
_save(withu_split, "withu_domain")                               # has 'split' column
for s in ["domain_train","domain_val","domain_test"]:
    _save(withu_split[withu_split.split==s], f"withu_{s}")
_save(withu_full[withu_full.quarantine], "withu_quarantine")     # for human re-review
print("Done.")

Writing to /home/yonsei/Downloads/prepared
  saved prepared/stage1_public.parquet  (117206 rows)
  saved prepared/withu_domain.parquet  (51434 rows)
  saved prepared/withu_domain_train.parquet  (36166 rows)
  saved prepared/withu_domain_val.parquet  (7637 rows)
  saved prepared/withu_domain_test.parquet  (7631 rows)
  saved prepared/withu_quarantine.parquet  (609 rows)
Done.


## 5. What you have now, and what's next

**Outputs in `./prepared/`:**
- `stage1_public.*` — BEEP + K-MHaS (+ AI-Hub once added), split `stage1_{train,val,test}` → **Phase 1 Stage-1** general-toxicity training.
- `withu_domain.*` (+ per-split files) — event-level `domain_{train,val,test}` → **Phase 1 Stage-2** domain adaptation and your **evaluation gold**.
- `withu_quarantine.*` — `검토`/`판정불가` rows parked for a second human pass.

**Phase 1 next steps (separate notebook):**
1. Tokenize with **KcELECTRA** (`beomi/KcELECTRA-base`) / **KoBERT**.
2. **Stage-1**: fine-tune the binary toxicity head on `stage1_train`, early-stop on `stage1_val`.
3. **Stage-2**: continue on `withu_domain_train`; evaluate precision/recall/F1 on `withu_domain_test`.
4. Add the **RAG slang layer** (researcher-maintained 은어/초성체 DB) at inference.

**Data-reality reminders baked into the split:** the domain test/val each rest on a single WCB event,
so treat those numbers as directional; the public test set is your powered check for the text head;
and the visual / exclusion / bystander modules stay data-starved until Phases 3–4 add collected +
synthetic events.
